# Measles Risk Modeling at County Level

## Overview
This notebook builds a machine learning model to predict measles occurrence risk at the county level using:
- **MMR vaccination coverage data** (primary predictor)
- **Facebook Social Connectedness Index** (network/transmission risk)
- **Geographic and demographic features**

## Modeling Approach
We use a **binary classification** framework to predict whether a county will experience measles cases.

### Why This Approach?
1. **Epidemiologically grounded**: Low vaccination coverage creates susceptible populations
2. **Network effects matter**: Social connectivity predicts disease transmission pathways
3. **Actionable outputs**: Identifies high-risk counties for targeted public health interventions

---

## Step 1: Import Required Libraries

We use standard data science libraries:
- `pandas`: Data manipulation
- `numpy`: Numerical operations
- `scikit-learn`: Machine learning models and evaluation
- `matplotlib`/`seaborn`: Visualization

In [ ]:
# Standard data manipulation libraries
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning - Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# Machine Learning - Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# Machine Learning - Evaluation
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    f1_score
)

# For handling class imbalance
from sklearn.utils.class_weight import compute_class_weight

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Set random seed for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Libraries imported successfully!")

---
## Step 2: Load and Explore the Data

We have three main data sources:
1. **Measles cases time series** - Our target variable source
2. **MMR vaccination coverage** - Primary predictor
3. **Facebook SCI** - Social connectivity features (loaded separately due to size)

In [ ]:
# Define file paths
DATA_DIR = Path('.')

# Load measles cases data
measles_df = pd.read_csv(DATA_DIR / 'measles_timeseries.csv')
print(f"Measles data shape: {measles_df.shape}")
print(f"Columns: {measles_df.columns.tolist()}")

In [ ]:
# Display first few rows of measles data
measles_df.head()

In [ ]:
# Load MMR vaccination coverage data
mmr_df = pd.read_csv(DATA_DIR / 'MMR_2026 - mmr.csv')
print(f"MMR data shape: {mmr_df.shape}")
print(f"Columns: {mmr_df.columns.tolist()}")

In [ ]:
# Display first few rows of MMR data
mmr_df.head()

### 2.1 Data Quality Assessment

In [ ]:
# Check for missing values in key columns
print("=== Measles Data Missing Values ===")
print(measles_df[['fips', 'county', 'state', 'cases_total', 'population', 'mmr_rate']].isnull().sum())
print("\n=== MMR Data Missing Values ===")
print(mmr_df.isnull().sum())

In [ ]:
# Check unique counties in each dataset
print(f"Unique FIPS codes in measles data: {measles_df['fips'].nunique()}")
print(f"Unique FIPS codes in MMR data: {mmr_df['FIPS'].nunique()}")

---
## Step 3: Data Preprocessing and Feature Engineering

### 3.1 Create the Target Variable

We need to identify which counties have had measles cases. Our target:
- **1** = County had at least one measles case
- **0** = County had no measles cases

**IMPORTANT DATA NOTE:** The measles time series data is **cumulative** - each row represents the total cases *up to that date* for a county. Therefore:
- We need to extract the **final (most recent) record** for each county to get the true total case count
- Using `max()` on cumulative data is equivalent to getting the final value
- We sort by date first to ensure we're capturing the correct temporal order

In [ ]:
# =============================================================================
# HANDLING CUMULATIVE DATA
# =============================================================================
# The measles data is CUMULATIVE - each row shows total cases up to that date.
# We need to get the LATEST record for each county to get the final case count.

# First, filter to county-level data only (exclude state-level aggregations)
county_measles = measles_df[measles_df['vaccination granularity'] == 'county'].copy()

# Convert date column to datetime for proper sorting
county_measles['date_parsed'] = pd.to_datetime(county_measles['date'], format='%m/%d/%y', errors='coerce')

# Sort by date to ensure we can get the latest record
county_measles = county_measles.sort_values('date_parsed')

print(f"Date range in data: {county_measles['date_parsed'].min()} to {county_measles['date_parsed'].max()}")
print(f"Total county-level records: {len(county_measles)}")

# Method 1: Get the LAST (most recent) record for each county
# This is the correct way to handle cumulative data
latest_records = county_measles.groupby(['fips', 'county', 'state']).last().reset_index()

# Method 2 (verification): Using max() should give same result for cumulative data
max_records = county_measles.groupby(['fips', 'county', 'state']).agg({
    'cases_total': 'max',
    'population': 'first',
    'longitude': 'first',
    'latitude': 'first',
    'mmr_rate': 'first',
    'imported': 'max',
    'local': 'max',
    'unvaccinated': 'max',
    'vaccinated': 'max'
}).reset_index()

# Use the latest records approach (more explicit about cumulative handling)
county_cases = latest_records[['fips', 'county', 'state', 'cases_total', 'population', 
                               'longitude', 'latitude', 'mmr_rate', 'imported', 
                               'local', 'unvaccinated', 'vaccinated']].copy()

print(f"\nUnique counties with measles cases: {len(county_cases)}")
print(f"Total cumulative cases across all counties: {county_cases['cases_total'].sum():,}")
print(f"\nTop 10 counties by case count:")
print(county_cases.nlargest(10, 'cases_total')[['county', 'state', 'cases_total']].to_string(index=False))

In [ ]:
# Standardize FIPS codes to 5-digit format
# Some FIPS codes may be stored without leading zeros

def standardize_fips(fips):
    """Convert FIPS code to standard 5-digit string format."""
    try:
        fips_int = int(float(fips))
        return str(fips_int).zfill(5)
    except (ValueError, TypeError):
        return None

# Apply to both datasets
mmr_df['fips_std'] = mmr_df['FIPS'].apply(standardize_fips)
county_cases['fips_std'] = county_cases['fips'].apply(standardize_fips)

print(f"MMR counties with valid FIPS: {mmr_df['fips_std'].notna().sum()}")
print(f"Measles counties with valid FIPS: {county_cases['fips_std'].notna().sum()}")

### 3.2 Create the Master County Dataset

We'll use the MMR dataset as the base (it has more counties) and merge in measles case information.

In [ ]:
# Start with MMR data as the base
# Select the most recent MMR rate available for each county

# Define school year columns in order (most recent first)
mmr_year_cols = [
    'SY2024_25', 'SY2023_24', 'SY2022_23', 'SY2021_22',
    'SY2020_21', 'SY2019_20', 'SY2018_19', 'SY2017_18'
]

def get_latest_mmr(row):
    """Get the most recent non-null MMR rate for a county."""
    for col in mmr_year_cols:
        if pd.notna(row[col]):
            return row[col]
    return np.nan

def get_mmr_trend(row):
    """Calculate the trend in MMR rates (difference between most recent and oldest available)."""
    rates = [row[col] for col in mmr_year_cols if pd.notna(row[col])]
    if len(rates) >= 2:
        return rates[0] - rates[-1]  # Recent minus oldest (positive = improving)
    return 0

def count_mmr_years(row):
    """Count how many years of MMR data are available."""
    return sum(pd.notna(row[col]) for col in mmr_year_cols)

# Create features from MMR data
mmr_df['latest_mmr'] = mmr_df.apply(get_latest_mmr, axis=1)
mmr_df['mmr_trend'] = mmr_df.apply(get_mmr_trend, axis=1)
mmr_df['mmr_years_available'] = mmr_df.apply(count_mmr_years, axis=1)

print(f"Counties with latest MMR rate: {mmr_df['latest_mmr'].notna().sum()}")
print(f"MMR rate statistics:")
print(mmr_df['latest_mmr'].describe())

In [ ]:
# Create the master dataset by starting with MMR data
master_df = mmr_df[['fips_std', 'County', 'State', 'latest_mmr', 'mmr_trend', 'mmr_years_available']].copy()
master_df = master_df.rename(columns={'County': 'county', 'State': 'state'})

# Create a set of counties that had measles cases
counties_with_cases = set(county_cases['fips_std'].dropna())

# Create target variable: 1 if county had cases, 0 otherwise
master_df['has_cases'] = master_df['fips_std'].apply(
    lambda x: 1 if x in counties_with_cases else 0
)

# Merge in additional features from county_cases where available
master_df = master_df.merge(
    county_cases[['fips_std', 'population', 'longitude', 'latitude', 'cases_total']],
    on='fips_std',
    how='left'
)

print(f"Master dataset shape: {master_df.shape}")
print(f"\nTarget variable distribution:")
print(master_df['has_cases'].value_counts())
print(f"\nPercentage with cases: {master_df['has_cases'].mean()*100:.2f}%")

### 3.3 Add Additional Features

We'll create features that capture:
1. **Vaccination risk levels** (categorized MMR rates)
2. **State-level aggregates** (state risk context)
3. **Placeholder for SCI features** (to be added with full data)

In [ ]:
# Create vaccination risk categories based on herd immunity thresholds
# Measles requires ~95% coverage for herd immunity

def categorize_mmr_risk(rate):
    """Categorize MMR rate into risk levels based on epidemiological thresholds.
    
    Risk levels:
    - High risk: <85% (well below herd immunity)
    - Medium-High risk: 85-90%
    - Medium risk: 90-95%
    - Low risk: >=95% (herd immunity threshold for measles)
    """
    if pd.isna(rate):
        return 'unknown'
    elif rate < 0.85:
        return 'high_risk'
    elif rate < 0.90:
        return 'medium_high_risk'
    elif rate < 0.95:
        return 'medium_risk'
    else:
        return 'low_risk'

master_df['mmr_risk_category'] = master_df['latest_mmr'].apply(categorize_mmr_risk)

print("MMR Risk Category Distribution:")
print(master_df['mmr_risk_category'].value_counts())

In [ ]:
# Create state-level features
# Counties in states with more cases may be at higher risk

state_stats = master_df.groupby('state').agg({
    'has_cases': 'sum',  # Number of counties with cases in state
    'latest_mmr': 'mean',  # Average MMR rate in state
    'fips_std': 'count'  # Number of counties in state
}).rename(columns={
    'has_cases': 'state_counties_with_cases',
    'latest_mmr': 'state_avg_mmr',
    'fips_std': 'state_county_count'
})

# Calculate proportion of counties with cases per state
state_stats['state_case_proportion'] = (
    state_stats['state_counties_with_cases'] / state_stats['state_county_count']
)

# Merge back to master
master_df = master_df.merge(state_stats, on='state', how='left')

print("State-level features added:")
print(state_stats.sort_values('state_counties_with_cases', ascending=False).head(10))

### 3.4 Facebook Social Connectedness Index (SCI) Features

The SCI data captures social connections between counties. This is valuable because:
- **Disease spreads through social networks**
- **Counties connected to outbreak areas are at higher risk**

Below is a placeholder function to process SCI data when available:

In [ ]:
def create_sci_features(sci_df, outbreak_counties, target_fips):
    """
    Create Social Connectedness Index features for a county.
    
    Parameters:
    -----------
    sci_df : pd.DataFrame
        Facebook SCI data with columns: user_region, friend_region, scaled_sci
    outbreak_counties : set
        Set of FIPS codes for counties with measles outbreaks
    target_fips : str
        FIPS code of the county to calculate features for
        
    Returns:
    --------
    dict : Dictionary of SCI-based features
    
    Features created:
    1. sci_to_outbreak_total: Sum of SCI to all outbreak counties
    2. sci_to_outbreak_max: Maximum SCI to any outbreak county
    3. sci_to_outbreak_count: Number of outbreak counties this county is connected to
    4. sci_weighted_outbreak_exposure: SCI-weighted exposure to outbreaks
    """
    # Filter SCI data for the target county
    county_connections = sci_df[
        (sci_df['user_region'] == target_fips) | 
        (sci_df['friend_region'] == target_fips)
    ]
    
    # Get connections to outbreak counties
    outbreak_connections = county_connections[
        county_connections['user_region'].isin(outbreak_counties) |
        county_connections['friend_region'].isin(outbreak_counties)
    ]
    
    return {
        'sci_to_outbreak_total': outbreak_connections['scaled_sci'].sum(),
        'sci_to_outbreak_max': outbreak_connections['scaled_sci'].max() if len(outbreak_connections) > 0 else 0,
        'sci_to_outbreak_count': len(outbreak_connections),
        'sci_weighted_outbreak_exposure': outbreak_connections['scaled_sci'].mean() if len(outbreak_connections) > 0 else 0
    }

# NOTE: To use this function with full SCI data:
# 1. Load the SCI CSV file (59MB)
# 2. For each county in master_df, call create_sci_features()
# 3. Add the resulting features to master_df

# Example usage (commented out due to data size):
# sci_df = pd.read_csv('path/to/sci_data.csv')
# sci_features = master_df['fips_std'].apply(
#     lambda x: create_sci_features(sci_df, counties_with_cases, x)
# )
# master_df = pd.concat([master_df, pd.DataFrame(sci_features.tolist())], axis=1)

print("SCI feature engineering function defined.")
print("To use: Load full SCI data and run create_sci_features() for each county.")

---
## Step 4: Prepare Data for Modeling

### 4.1 Select Features and Handle Missing Values

In [ ]:
# Define feature columns for modeling
# We'll use features that are available for most counties

feature_cols = [
    'latest_mmr',           # Most important: vaccination rate
    'mmr_trend',            # Is vaccination improving or declining?
    'mmr_years_available',  # Data completeness indicator
    'state_avg_mmr',        # State context
    'state_case_proportion' # State outbreak context
]

# Check data availability
print("Feature availability:")
for col in feature_cols:
    available = master_df[col].notna().sum()
    pct = available / len(master_df) * 100
    print(f"  {col}: {available} ({pct:.1f}%)")

In [ ]:
# Create modeling dataset
# Remove rows with missing target or all-missing features

model_df = master_df.dropna(subset=['latest_mmr']).copy()
print(f"Counties with valid MMR data for modeling: {len(model_df)}")

# Prepare features (X) and target (y)
X = model_df[feature_cols].copy()
y = model_df['has_cases'].copy()

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nTarget distribution:")
print(y.value_counts())

In [ ]:
# Handle any remaining missing values
# Using median imputation for robustness

imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns,
    index=X.index
)

print("Missing values after imputation:")
print(X_imputed.isnull().sum())

### 4.2 Train-Test Split

We use stratified sampling to preserve the class distribution in both train and test sets.

In [ ]:
# Split data into training and testing sets
# Using 80-20 split with stratification to handle class imbalance

X_train, X_test, y_train, y_test = train_test_split(
    X_imputed, y, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=y  # Ensures same class proportion in train and test
)

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")
print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTest set class distribution:")
print(y_test.value_counts())

In [ ]:
# Scale features for models that benefit from normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame for easier interpretation
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print("Features scaled successfully.")

### 4.3 Handle Class Imbalance

Since measles cases are relatively rare (few counties have cases), we need to address class imbalance.

In [ ]:
# Calculate class weights to handle imbalance
# This gives more importance to the minority class (counties with cases)

classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

print(f"Class weights: {class_weight_dict}")
print(f"\nInterpretation: Class 1 (has cases) is weighted {class_weight_dict[1]/class_weight_dict[0]:.2f}x more than class 0")

---
## Step 5: Model Training and Evaluation

We'll train multiple models and compare their performance:
1. **Logistic Regression** - Simple, interpretable baseline
2. **Random Forest** - Handles non-linear relationships
3. **Gradient Boosting** - Often best performance

### 5.1 Define Evaluation Functions

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):
    """
    Comprehensive model evaluation with multiple metrics.
    
    Parameters:
    -----------
    model : trained sklearn model
    X_test : test features
    y_test : true labels
    model_name : string for display purposes
    
    Returns:
    --------
    dict : Dictionary of evaluation metrics
    """
    # Get predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    metrics = {
        'model': model_name,
        'roc_auc': roc_auc_score(y_test, y_prob),
        'avg_precision': average_precision_score(y_test, y_prob),
        'f1': f1_score(y_test, y_pred)
    }
    
    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"{'='*50}")
    print(f"\nROC-AUC Score: {metrics['roc_auc']:.4f}")
    print(f"Average Precision: {metrics['avg_precision']:.4f}")
    print(f"F1 Score: {metrics['f1']:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['No Cases', 'Has Cases']))
    print(f"\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    return metrics, y_prob

### 5.2 Train Models

In [ ]:
# Dictionary to store results
results = {}
probabilities = {}

# 1. Logistic Regression
# Good baseline, highly interpretable
print("Training Logistic Regression...")
lr_model = LogisticRegression(
    class_weight='balanced',  # Handle imbalance
    max_iter=1000,
    random_state=RANDOM_STATE
)
lr_model.fit(X_train_scaled, y_train)
results['Logistic Regression'], probabilities['Logistic Regression'] = evaluate_model(
    lr_model, X_test_scaled, y_test, 'Logistic Regression'
)

In [ ]:
# 2. Random Forest
# Captures non-linear relationships, robust to outliers
print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)  # RF doesn't need scaling
results['Random Forest'], probabilities['Random Forest'] = evaluate_model(
    rf_model, X_test, y_test, 'Random Forest'
)

In [ ]:
# 3. Gradient Boosting
# Often achieves best performance, good with imbalanced data
print("Training Gradient Boosting...")
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=RANDOM_STATE
)
gb_model.fit(X_train, y_train)
results['Gradient Boosting'], probabilities['Gradient Boosting'] = evaluate_model(
    gb_model, X_test, y_test, 'Gradient Boosting'
)

### 5.3 Model Comparison

In [ ]:
# Compare all models
comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df.sort_values('roc_auc', ascending=False)

print("\n" + "="*60)
print("MODEL COMPARISON SUMMARY")
print("="*60)
print(comparison_df.to_string())

In [ ]:
# Plot ROC curves for all models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curve
ax1 = axes[0]
for model_name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax1.plot(fpr, tpr, label=f'{model_name} (AUC={auc:.3f})')

ax1.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curves - Model Comparison')
ax1.legend(loc='lower right')
ax1.grid(True, alpha=0.3)

# Precision-Recall Curve
ax2 = axes[1]
for model_name, y_prob in probabilities.items():
    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    ap = average_precision_score(y_test, y_prob)
    ax2.plot(recall, precision, label=f'{model_name} (AP={ap:.3f})')

ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curves')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved as 'model_comparison_curves.png'")

---
## Step 6: Feature Importance Analysis

Understanding which features drive predictions helps validate the model epidemiologically.

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Feature Importance (Random Forest):")
print(feature_importance.to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=feature_importance, x='importance', y='feature', ax=ax, palette='viridis')
ax.set_xlabel('Importance')
ax.set_ylabel('Feature')
ax.set_title('Feature Importance for Measles Risk Prediction')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved as 'feature_importance.png'")

In [ ]:
# Logistic Regression coefficients (more interpretable)
lr_coefficients = pd.DataFrame({
    'feature': feature_cols,
    'coefficient': lr_model.coef_[0],
    'odds_ratio': np.exp(lr_model.coef_[0])
}).sort_values('coefficient', key=abs, ascending=False)

print("\nLogistic Regression Coefficients:")
print(lr_coefficients.to_string(index=False))
print("\nInterpretation:")
print("- Negative coefficient for 'latest_mmr' means higher vaccination = lower risk (expected!)")
print("- Odds ratio < 1 means that feature is protective")
print("- Odds ratio > 1 means that feature increases risk")

---
## Step 7: Cross-Validation for Robust Estimates

Single train-test splits can be unstable. Cross-validation gives more reliable performance estimates.

In [ ]:
# Perform stratified k-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_results = {}
for name, model in [('Logistic Regression', lr_model), 
                    ('Random Forest', rf_model),
                    ('Gradient Boosting', gb_model)]:
    # Use appropriate features (scaled for LR, unscaled for tree-based)
    X_cv = X_imputed if 'Logistic' not in name else pd.DataFrame(
        StandardScaler().fit_transform(X_imputed), columns=feature_cols
    )
    
    scores = cross_val_score(model, X_cv, y, cv=cv, scoring='roc_auc')
    cv_results[name] = {
        'mean_auc': scores.mean(),
        'std_auc': scores.std(),
        'scores': scores
    }
    print(f"{name}: AUC = {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

---
## Step 8: Generate Risk Predictions for All Counties

Apply the best model to generate risk scores for all counties.

In [ ]:
# Select best model based on cross-validation
best_model_name = max(cv_results, key=lambda x: cv_results[x]['mean_auc'])
print(f"Best model: {best_model_name}")

# Use the appropriate model
if best_model_name == 'Random Forest':
    best_model = rf_model
    X_pred = X_imputed
elif best_model_name == 'Gradient Boosting':
    best_model = gb_model
    X_pred = X_imputed
else:
    best_model = lr_model
    X_pred = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_cols)

In [ ]:
# Generate risk predictions for all counties
model_df['risk_probability'] = best_model.predict_proba(X_pred)[:, 1]
model_df['risk_category'] = pd.cut(
    model_df['risk_probability'],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

print("Risk Category Distribution:")
print(model_df['risk_category'].value_counts())

In [ ]:
# Show highest risk counties WITHOUT current cases (most actionable for public health)
high_risk_no_cases = model_df[
    (model_df['has_cases'] == 0) & 
    (model_df['risk_probability'] > 0.5)
].sort_values('risk_probability', ascending=False)

print(f"\nHigh-Risk Counties WITHOUT Current Cases (Top 20):")
print(high_risk_no_cases[['county', 'state', 'latest_mmr', 'risk_probability', 'risk_category']].head(20).to_string(index=False))

In [ ]:
# Save predictions to CSV
output_cols = ['fips_std', 'county', 'state', 'latest_mmr', 'has_cases', 
               'risk_probability', 'risk_category']
model_df[output_cols].to_csv('county_risk_predictions.csv', index=False)
print("\nPredictions saved to 'county_risk_predictions.csv'")

---
## Step 9: Model Validation with Actual Cases

Check how well the model's risk scores align with actual case occurrence.

In [ ]:
# Analyze relationship between predicted risk and actual cases
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot of risk scores by actual case status
ax1 = axes[0]
model_df.boxplot(column='risk_probability', by='has_cases', ax=ax1)
ax1.set_xlabel('Has Measles Cases')
ax1.set_ylabel('Predicted Risk Probability')
ax1.set_title('Risk Score Distribution by Actual Case Status')
plt.suptitle('')  # Remove automatic title

# Distribution of MMR rates by case status
ax2 = axes[1]
for has_case in [0, 1]:
    subset = model_df[model_df['has_cases'] == has_case]['latest_mmr']
    label = 'Has Cases' if has_case == 1 else 'No Cases'
    ax2.hist(subset, bins=20, alpha=0.6, label=label, density=True)
ax2.axvline(x=0.95, color='red', linestyle='--', label='Herd Immunity Threshold (95%)')
ax2.set_xlabel('MMR Vaccination Rate')
ax2.set_ylabel('Density')
ax2.set_title('MMR Rate Distribution by Case Status')
ax2.legend()

plt.tight_layout()
plt.savefig('model_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved as 'model_validation.png'")

In [ ]:
# Statistical summary
print("\n" + "="*60)
print("SUMMARY STATISTICS")
print("="*60)

print("\nMMR Rate by Case Status:")
print(model_df.groupby('has_cases')['latest_mmr'].describe())

print("\nRisk Probability by Case Status:")
print(model_df.groupby('has_cases')['risk_probability'].describe())

---
## Step 10: Conclusions and Next Steps

### Key Findings:

1. **Vaccination coverage is a strong predictor** of measles risk
2. **State-level context matters** - counties in states with outbreaks are at higher risk
3. **The model can identify high-risk counties** that haven't yet experienced cases

### Limitations:

1. **Class imbalance**: Relatively few counties have cases
2. **Missing data**: Not all counties have MMR coverage data
3. **Temporal dynamics**: Current model is static; outbreaks evolve over time
4. **SCI data not integrated**: Full social connectivity analysis pending

### Recommended Next Steps:

1. **Integrate Facebook SCI data** to capture network transmission risk
2. **Add temporal features** to model outbreak dynamics
3. **Include demographic covariates** (age distribution, population density)
4. **Consider spatial models** to account for geographic clustering
5. **Validate prospectively** as new cases emerge

In [ ]:
# Final summary
print("\n" + "="*60)
print("MEASLES RISK MODELING - FINAL SUMMARY")
print("="*60)
print(f"\nTotal counties analyzed: {len(model_df)}")
print(f"Counties with measles cases: {model_df['has_cases'].sum()}")
print(f"Counties without cases: {(model_df['has_cases']==0).sum()}")
print(f"\nBest Model: {best_model_name}")
print(f"Cross-Validation AUC: {cv_results[best_model_name]['mean_auc']:.4f}")
print(f"\nHigh-risk counties identified (no current cases): {len(high_risk_no_cases)}")
print("\nOutput files generated:")
print("  - county_risk_predictions.csv")
print("  - model_comparison_curves.png")
print("  - feature_importance.png")
print("  - model_validation.png")